# **Training Notebook**

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import sklearn
import os
import utils

## Data loading

In [2]:
# Define the path as required by the guide
data_path = os.path.expanduser("~/Datasets/QuoraQuestionPairs/quora_data.csv")

quora_df = pd.read_csv(data_path)

A_df, test_df = train_test_split(
    quora_df, 
    test_size=0.05, 
    random_state=123
)

train_df, val_df = train_test_split(
    A_df, 
    test_size=0.05, 
    random_state=123
)

print(f'train_df.shape = {train_df.shape}')
print(f'val_df.shape = {val_df.shape}')
print(f'test_df.shape = {test_df.shape}')

display(train_df)
display(train_df.dtypes)

train_df.shape = (291897, 6)
val_df.shape = (15363, 6)
test_df.shape = (16172, 6)


,id,qid1,qid2,question1,question2,is_duplicate
61482,125898,203030,203031,Is Java or C++ or C the most popular language ...,How do I develop a software which will have a ...,0
131546,36249,66113,66114,How do you convert direct speech into reported...,I feel weak at spoken English. I have sentence...,0
22927,199864,301469,301470,Where can I buy used wine barrels?,Where can you buy used wine barrels?,1
183520,277339,17728,138400,What was the best day of your life? (Excluding...,What is the Best Day of your life till date?,1
67694,392907,525647,525648,How is web-work.in works?,How do I get web designing work?,0
...,...,...,...,...,...,...
231903,206882,310426,310427,How many views I have to have on YouTube to ea...,"How many views, likes, and comments should a Y...",1
115066,298165,174701,420621,Are nihilist atheists?,Are all nihilist atheists?,1
35776,98943,164337,164338,How does views of sentiments of people relate ...,Does the increased mobile use of Quora as a pl...,0
300894,18300,34679,34680,What are the pros and cons of owning an EV (el...,Where is Mount Villarica located and how does ...,0


id               int64
qid1             int64
qid2             int64
question1       object
question2       object
is_duplicate     int64
dtype: object

## Baseline model

In [3]:
# Transform questions to strings (if not we may find some errors)
q1_train =  utils.cast_list_as_strings(list(train_df["question1"]))
q2_train =  utils.cast_list_as_strings(list(train_df["question2"]))

# Fitting a Count Vectorizer
count_vectorizer = sklearn.feature_extraction.text.CountVectorizer(ngram_range=(1,1))
count_vectorizer.fit(q1_train + q2_train)

X_tr_q1q2 = utils.get_features_from_df(train_df, count_vectorizer)

logistic = sklearn.linear_model.LogisticRegression(solver="liblinear",
                                                   random_state=123)
y_train = train_df["is_duplicate"].values
logistic.fit(X_tr_q1q2, y_train)

LogisticRegression(random_state=123, solver='liblinear')

In [4]:
y_train = train_df["is_duplicate"].values

mistake_indices, predictions = utils.get_mistakes(logistic, X_tr_q1q2, y_train)
print(f"Accuracy: {(len(predictions) - len(mistake_indices)) / len(predictions) * 100}%\n")

def print_mistake_k(k, mistake_indices, predictions):
    print(train_df.iloc[mistake_indices[k]].question1)
    print(train_df.iloc[mistake_indices[k]].question2)
    print("true class:", train_df.iloc[mistake_indices[k]].is_duplicate)
    print("prediction:", predictions[mistake_indices[k]])
    print()

print_mistake_k(99, mistake_indices, predictions)
print_mistake_k(41, mistake_indices, predictions)

Accuracy: 81.39960328472029%

What songs make you cry?
What are some songs that make you cry?
true class: 1
prediction: 0

What is climate change and why is it important?
Is Cambodia experiencing climate-change?
true class: 0
prediction: 1



## Models directory

The guide requires that models are **only saved if the `models/` folder does not already exist**.

In [5]:
# ADDED
MODELS_DIR = "models"
ALREADY_TRAINED = os.path.exists(MODELS_DIR)

if not ALREADY_TRAINED:
    os.makedirs(MODELS_DIR)
    print(f"Created '{MODELS_DIR}/' — models will be trained and saved.")
else:
    print(f"'{MODELS_DIR}/' already exists — training will be skipped for any model already on disk.")


Created 'models/' — models will be trained and saved.


## Save baseline model

Persist the CountVectorizer and the baseline Logistic Regression trained above.

In [6]:
# ADDED
from utils import save_object, load_object

CV_PATH       = os.path.join(MODELS_DIR, "count_vectorizer.pkl")
BASELINE_PATH = os.path.join(MODELS_DIR, "baseline_logistic.pkl")

if not os.path.exists(CV_PATH):
    save_object(count_vectorizer, CV_PATH)
    print("Saved count_vectorizer.pkl")
else:
    print("count_vectorizer.pkl already exists — skipping.")

if not os.path.exists(BASELINE_PATH):
    save_object(logistic, BASELINE_PATH)
    print("Saved baseline_logistic.pkl")
else:
    print("baseline_logistic.pkl already exists — skipping.")


Saved count_vectorizer.pkl
Saved baseline_logistic.pkl


## Improved model

We augment the BoW representation with **five handcrafted features** computed from scratch:

| # | Feature | Implementation |
|---|---------|----------------|
| 0 | Jaccard similarity (word sets) | from scratch |
| 1 | Length ratio (min/max) | from scratch |
| 2 | Common-word F1 | from scratch |
| 3 | Character-bigram Dice coefficient | from scratch |
| 4 | TF-IDF cosine similarity | from scratch (vectorised) |

A TF-IDF vectorizer (fitted on train only) is used for feature 4.

In [7]:
# ADDED
from utils import get_combined_features, get_handcrafted_features, cosine_similarity_tfidf_batch

TFIDF_PATH    = os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl")
IMPROVED_PATH = os.path.join(MODELS_DIR, "improved_logistic.pkl")

if not os.path.exists(TFIDF_PATH):
    # Fit TF-IDF on training questions only (no data leakage)  # ADDED
    q1_train_str = utils.cast_list_as_strings(list(train_df["question1"]))
    q2_train_str = utils.cast_list_as_strings(list(train_df["question2"]))
    tfidf_vectorizer = sklearn.feature_extraction.text.TfidfVectorizer(
        ngram_range=(1, 1), min_df=3, sublinear_tf=True
    )
    tfidf_vectorizer.fit(q1_train_str + q2_train_str)
    save_object(tfidf_vectorizer, TFIDF_PATH)
    print(f"TF-IDF vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")
    print("Saved tfidf_vectorizer.pkl")
else:
    tfidf_vectorizer = load_object(TFIDF_PATH)
    print("Loaded existing tfidf_vectorizer.pkl")


TF-IDF vocabulary size: 34428
Saved tfidf_vectorizer.pkl


In [8]:
# ADDED — build combined feature matrix for training
if not os.path.exists(IMPROVED_PATH):
    print("Building combined features for train set (this may take a few minutes)...")
    X_tr_combined = get_combined_features(train_df, count_vectorizer, tfidf_vectorizer)
    print(f"X_tr_combined.shape = {X_tr_combined.shape}")

    improved_logistic = sklearn.linear_model.LogisticRegression(
        solver="liblinear", random_state=123
    )
    improved_logistic.fit(X_tr_combined, y_train)
    save_object(improved_logistic, IMPROVED_PATH)
    print("Saved improved_logistic.pkl")
else:
    print("improved_logistic.pkl already exists — skipping training.")


Building combined features for train set (this may take a few minutes)...


/home/arnau/Master Data Science/NLP/NLP_quora_challenge/utils.py:128: RuntimeWarning: invalid value encountered in divide
  cosine_sims = np.where(denom > 0, dot_products / denom, 0.0)  # ADDED


X_tr_combined.shape = (291897, 149655)


Saved improved_logistic.pkl


## Quick sanity check on validation set

In [9]:
# ADDED
from utils import evaluate_model

# Reload improved model in case this cell is run after skipping training  # ADDED
if not os.path.exists(IMPROVED_PATH):
    print("Improved model not found — run training cells first.")
else:
    _improved = load_object(IMPROVED_PATH)
    _tv       = load_object(TFIDF_PATH)
    _cv       = load_object(CV_PATH)
    _base     = load_object(BASELINE_PATH)

    X_val_bow      = utils.get_features_from_df(val_df, _cv)
    X_val_combined = get_combined_features(val_df, _cv, _tv)
    y_val          = val_df["is_duplicate"].values

    import pandas as pd
    results = [
        evaluate_model(_base,     X_val_bow,      y_val, "baseline",  "val"),
        evaluate_model(_improved, X_val_combined, y_val, "improved",  "val"),
    ]
    display(pd.DataFrame(results))


/home/arnau/Master Data Science/NLP/NLP_quora_challenge/utils.py:128: RuntimeWarning: invalid value encountered in divide
  cosine_sims = np.where(denom > 0, dot_products / denom, 0.0)  # ADDED


,model,split,roc_auc,precision,recall,f1
0,baseline,val,0.8046,0.6772,0.6107,0.6422
1,improved,val,0.8781,0.7267,0.7129,0.7198
